# 🎙️ AI Meeting Assistant — run on Google Colab (GPU)

WhisperX (Whisper + word alignment + speaker labels) on a **Colab GPU**, then **Qwen2.5-72B-Instruct** (via OpenRouter) for transcript refinement and for minutes / decisions / action items.

**How to run (about 10 minutes the first time, models download once):**
1. **Runtime ▸ Change runtime type ▸ T4 GPU ▸ Save.**
2. *(Optional)* click the 🔑 **Secrets** icon in the left bar, add `OPENROUTER_API_KEY` and `HF_TOKEN`, and switch on *Notebook access*. If you skip this, Step 4 asks you to paste them.
3. **Runtime ▸ Run all.**
4. Click the link printed under **Step 6**, upload a recording, press **▶ Process recording**.

Keep this tab open while using the app. If Colab disconnects, the link stops working: just run the notebook again.

## Step 1 · Check the GPU

In [ ]:
import subprocess, shutil
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
if r.returncode == 0 and r.stdout.strip():
    print("✅ GPU:", r.stdout.strip())
else:
    print("⚠️ No GPU found. Runtime ▸ Change runtime type ▸ T4 GPU, then Runtime ▸ Restart session and run all.\n"
          "   (It still works on CPU, but it is many times slower.)")
if not shutil.which("ffmpeg"):
    subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, check=True)
print("ffmpeg:", shutil.which("ffmpeg"))

## Step 2 · Get the code
Set `REPO_URL` to your GitHub repository (the folder that contains `app.py`, or its parent). Leave it empty to upload the project `.zip` instead.

In [ ]:
REPO_URL = ""      # e.g. "https://github.com/<your-user>/<your-repo>.git"   (empty = upload the zip)
BRANCH   = "main"

import os, glob, shutil, subprocess, zipfile
os.chdir("/content")
shutil.rmtree("/content/src", ignore_errors=True)
if REPO_URL.strip():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL.strip(), "/content/src"], check=True)
else:
    from google.colab import files
    print("Choose the project zip (e.g. meeting_assistant_v3_whisperx.zip)")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall("/content/src")
hits = sorted(glob.glob("/content/src/**/app.py", recursive=True), key=len)
assert hits, "app.py was not found - check REPO_URL or the zip you uploaded"
APP_DIR = os.path.dirname(hits[0])
print("✅ App folder:", APP_DIR)

## Step 3 · Install dependencies (about 3–5 minutes)
Uses the CUDA build of PyTorch that WhisperX needs (same versions as the project README). Warnings about other Colab packages are normal.

In [ ]:
!pip install -q torch==2.8.0 torchaudio==2.8.0 torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu128
!pip install -q -r "{APP_DIR}/requirements.txt"

In [ ]:
# sanity check in a fresh interpreter (so the notebook kernel stays clean)
!python -c "import torch, whisperx; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())"

## Step 4 · Keys
* **OpenRouter key** (required): https://openrouter.ai/keys
* **Hugging Face READ token** (optional, only for *speaker labels*): https://huggingface.co/settings/tokens. The same account must also click **“Agree and access repository”** on https://huggingface.co/pyannote/speaker-diarization-community-1. Leave blank to run without speaker labels.

In [ ]:
import os, getpass

def get_secret(name, prompt):
    val = os.environ.get(name, "")
    if not val:
        try:
            from google.colab import userdata          # Colab 🔑 Secrets panel
            val = userdata.get(name) or ""
        except Exception:
            val = ""
    if not val:
        val = getpass.getpass(f"{prompt} (press Enter to skip): ").strip()
    return val

OPENROUTER_API_KEY = get_secret("OPENROUTER_API_KEY", "OpenRouter API key")
HF_TOKEN           = get_secret("HF_TOKEN", "Hugging Face READ token")
assert OPENROUTER_API_KEY, "An OpenRouter API key is required for the language-model stages."

if HF_TOKEN:
    try:
        from huggingface_hub import HfApi
        HfApi().auth_check("pyannote/speaker-diarization-community-1", token=HF_TOKEN)
        print("✅ Hugging Face token OK - speaker labels available")
    except AttributeError:
        print("ℹ️ Token stored (could not pre-check access with this huggingface_hub version)")
    except Exception as e:
        print(f"⚠️ No access to the speaker model yet ({type(e).__name__}). Accept the terms on the model page; "
              "until then the app runs without speaker labels.")
else:
    print("ℹ️ No HF token - the app will run without speaker labels")

## Step 5 · Settings
The language model is set here (no code edits needed). Change `LLM_MODEL` if you want a different OpenRouter model.

In [ ]:
LLM_MODEL = "qwen/qwen-2.5-72b-instruct"

APP_ENV = os.environ.copy()
APP_ENV.update({
    "OPENROUTER_API_KEY": OPENROUTER_API_KEY,
    "HF_TOKEN": HF_TOKEN,
    "LLM_BASE_URL": "https://openrouter.ai/api/v1",
    "LLM_MODEL": LLM_MODEL,
    "LLM_JSON_MODE": "json_object",
    "EXTRACT_SINGLE_PASS_CHARS": "48000",   # larger model -> bigger single pass
    "EXTRACT_CHUNK_CHARS": "36000",
    "WHISPERX_MODEL": "auto",               # GPU -> large-v3-turbo (float16); CPU -> small.en
    "WHISPERX_DEVICE": "auto",
    "PYTHONUNBUFFERED": "1",
})
print("LLM:", LLM_MODEL)

## Step 6 · Start the app and get the link
Prints a public `https://….trycloudflare.com` link. Open it, upload a recording and press **▶ Process recording**.
Running this cell again restarts the app (and gives a new link).

In [ ]:
import sys, time, re, urllib.request
from IPython.display import HTML, display

subprocess.run("pkill -f 'streamlit run'; pkill -f cloudflared", shell=True)
time.sleep(1)

# 1) Streamlit, started from the app folder so .streamlit/config.toml (2 GB upload limit) is used
st_log = open("/content/streamlit.log", "w")
subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
     "--browser.gatherUsageStats", "false"],
    cwd=APP_DIR, env=APP_ENV, stdout=st_log, stderr=subprocess.STDOUT)

for _ in range(90):
    try:
        if urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2).read().strip() == b"ok":
            break
    except Exception:
        time.sleep(1)
else:
    print(open("/content/streamlit.log").read()[-3000:])
    raise RuntimeError("Streamlit did not start - see the log above")
print("✅ Streamlit is running")

# 2) public tunnel (no account needed)
CF = "/content/cloudflared"
if not os.path.exists(CF):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", CF)
    os.chmod(CF, 0o755)
cf_log = open("/content/cloudflared.log", "w")
subprocess.Popen([CF, "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                 stdout=cf_log, stderr=subprocess.STDOUT)

PUBLIC_URL = None
for _ in range(60):
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if m:
        PUBLIC_URL = m.group(0)
        break
    time.sleep(1)

if PUBLIC_URL:
    display(HTML(f'<a href="{PUBLIC_URL}" target="_blank" style="font-size:22px;font-weight:600">'
                 f'▶ Open the AI Meeting Assistant</a><br><small>{PUBLIC_URL}</small>'))
else:
    print(open("/content/cloudflared.log").read()[-2000:])
    print("Could not create the link - use the backup cell below.")

---
## Extras (only if needed)

**A · Backup link** (if the Step 6 link does not appear). Open the printed URL and enter the printed IP as the password.

In [ ]:
import subprocess
print("Password (this machine's IP):", subprocess.run("curl -s https://ipv4.icanhazip.com", shell=True, capture_output=True, text=True).stdout.strip())
!npx --yes localtunnel --port 8501

**B · Very large recordings.** The public link limits a single browser upload to roughly 100 MB (a long uncompressed WAV can exceed that). Either convert to MP3/M4A first, or upload the file here and paste the printed path into the app's *“…or use a file already on this machine”* box.

In [ ]:
os.chdir("/content")
from google.colab import files
up = files.upload()
for name in up:
    print("Paste this path into the app:", os.path.abspath(name))

**C · Troubleshooting:** show the latest app log.

In [ ]:
print(open("/content/streamlit.log").read()[-4000:])